# 04 — Spatial Analysis & Equity Overlay

**Project:** Chicago Road Safety Investment Prioritizer  
**Aligned with:** City of Chicago Vision Zero goals  
**Type:** Read-only spatial GIS & equity analysis — no source files are modified.  
**Grain:** 43 High-Crash Corridors (spatial GIS linework, EPSG:3435 / EPSG:4326, CDC SVI equity tract overlay)  

---

This notebook analyzes the spatial geography, network length, crash density (crashes/mile),
and CDC Social Vulnerability Index (SVI) equity overlays across Chicago's 43 high-crash corridors.
All metrics are derived directly from the authoritative spatial serving datasets.


In [ ]:
from __future__ import annotations
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import geopandas as gpd
from shapely.wkt import loads as load_wkt

%matplotlib inline
plt.rcParams.update({
    'figure.dpi': 110,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.titlesize': 11,
    'axes.labelsize': 9,
})

ROOT = Path('.').resolve()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
print('Project root:', ROOT)


---
## Section 1 — Purpose & Method

### Spatial Methodology & GIS Standards
- **Spatial Grain:** 43 High-Crash Corridors defined in the City of Chicago High-Crash Corridor Framework Plan.
- **Coordinate Reference Systems (CRS):**
  - **EPSG:3435 (NAD83 / Illinois East, feet):** Authoritative projection for distance, length, and buffer calculations.
  - **EPSG:4326 (WGS84, lat/lon):** Geographic display projection for interactive mapping.
- **CDC/ATSDR Social Vulnerability Index (SVI) Overlay:** Corridors are overlaid with 2022 CDC SVI census tracts to calculate:
  - **Corridor Length-Weighted SVI:** $\bar{SVI}_c = \frac{\sum_t L_{c,t} \times SVI_t}{\sum_t L_{c,t}}$
  - **High-SVI Length Share:** Fraction of corridor length traversing tracts with SVI $\ge 0.75$.
- **Equity Priority Classification Rules:**
  - **Rule A (Primary):** Weighted SVI $\ge 0.75$.
  - **Rule B (Alternative):** High-SVI length share $\ge 50\%$.
- **Spatial Crash Assignment:** 100-foot buffer matching (nearest corridor within 100 ft).


---
## Section 2 — Corridor Length & Network Geometry Summary

Summary of GIS network length, spatial extent, geometry types, and projection metadata.


In [ ]:
from dashboard.streamlit.data_access import load_corridor_geodataframe, load_corridor_master

gdf_corridors = load_corridor_geodataframe()
df_master     = load_corridor_master()

tot_miles   = df_master['spatial_total_length_miles'].sum()
min_miles   = df_master['spatial_total_length_miles'].min()
mean_miles  = df_master['spatial_total_length_miles'].mean()
max_miles   = df_master['spatial_total_length_miles'].max()
geom_types  = gdf_corridors.geometry.geom_type.value_counts()

print('── Network Geometry Summary ────────────')
print('  Total network corridors  :', len(gdf_corridors))
print('  Total network mileage    :', f'{tot_miles:.2f} miles')
print('  Corridor length range    :', f'{min_miles:.2f} to {max_miles:.2f} miles (mean: {mean_miles:.2f} miles)')
print('  Geometry Types           :', dict(geom_types))
print('  Display Projection (CRS) :', gdf_corridors.crs)


---
## Section 3 — SVI Equity Overlay & Priority Classification

Evaluation of SVI scores, equity priority classification, and spatial distribution across the 43 corridors.


In [ ]:
rule_a_count = int(df_master['equity_classification_A_weighted_ge_0_75'].sum())
rule_b_count = int(df_master['equity_classification_B_share_ge_0_50'].sum())

print('── SVI Equity Priority Classification Summary ─────────────────')
print(f'  Rule A (Weighted SVI ≥ 0.75)       : {rule_a_count:>2} corridors ({rule_a_count/len(df_master):.1%})')
print(f'  Rule B (High-SVI Share ≥ 50%)       : {rule_b_count:>2} corridors ({rule_b_count/len(df_master):.1%})')
print()
print('Top 10 Highest SVI Corridors:')
top10_svi = df_master.sort_values('corridor_length_weighted_svi', ascending=False).head(10)
print(top10_svi[['corridor_id','corridor_name','spatial_total_length_miles','corridor_length_weighted_svi','high_svi_length_share','equity_classification_A_weighted_ge_0_75']].to_string(index=False))


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Left: Distribution of Length-Weighted SVI Scores
axes[0].hist(df_master['corridor_length_weighted_svi'], bins=20, color='#3498db', alpha=0.8, edgecolor='white')
axes[0].axvline(0.75, color='#e74c3c', linestyle='--', linewidth=1.5, label='Rule A Threshold (0.75)')
axes[0].set_title('Distribution of Corridor Length-Weighted SVI Scores')
axes[0].set_xlabel('Weighted SVI Score (0.0 to 1.0)')
axes[0].set_ylabel('Corridor Count')
axes[0].legend(fontsize=9)

# Right: Distribution of High-SVI Length Share
axes[1].hist(df_master['high_svi_length_share'] * 100, bins=20, color='#2ecc71', alpha=0.8, edgecolor='white')
axes[1].axvline(50, color='#e67e22', linestyle='--', linewidth=1.5, label='Rule B Threshold (50%)')
axes[1].set_title('Distribution of High-SVI Tract Length Share (%)')
axes[1].set_xlabel('High-SVI Length Share (%)')
axes[1].set_ylabel('Corridor Count')
axes[1].legend(fontsize=9)

plt.suptitle('Section 3 — SVI Equity Priority Overlay Distributions', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()


---
## Section 4 — Spatial Map Visualization

Plotting corridor centroids and linework categorized by equity priority status.


In [ ]:
fig, ax = plt.subplots(figsize=(9, 10))

# Plot linework
eq_mask = df_master['equity_classification_A_weighted_ge_0_75'] == True
gdf_corridors.plot(ax=ax, color='#bdc3c7', linewidth=1.5, alpha=0.6, label='All High-Crash Corridors')

# Plot centroids
ax.scatter(
    df_master[~eq_mask]['centroid_longitude'], df_master[~eq_mask]['centroid_latitude'],
    color='#3498db', s=50, alpha=0.85, label='Non-Equity Corridors (SVI < 0.75)', zorder=4
)
ax.scatter(
    df_master[eq_mask]['centroid_longitude'], df_master[eq_mask]['centroid_latitude'],
    color='#e67e22', s=70, alpha=0.9, label='Equity Priority Corridors (SVI ≥ 0.75)', zorder=5
)

for _, r in df_master[eq_mask].head(6).iterrows():
    ax.annotate(r['corridor_id'], (r['centroid_longitude'], r['centroid_latitude']),
                xytext=(5, 5), textcoords='offset points', fontsize=7, fontweight='bold', color='#d35400')

ax.set_title('Section 4 — Chicago High-Crash Corridor Network & Equity Overlay', fontsize=11, fontweight='bold')
ax.set_xlabel('Longitude (°W)')
ax.set_ylabel('Latitude (°N)')
ax.legend(fontsize=8, loc='upper left')
plt.tight_layout()
plt.show()


---
## Section 5 — Spatial Crash Density Analysis

Computing 2026 forecast crash density (total crashes per mile and KSI crashes per mile) to identify spatial risk bottlenecks.


In [ ]:
df_master['forecast_density_total_per_mile'] = df_master['annual_forecast_total_crashes_2026'] / df_master['spatial_total_length_miles']
df_master['forecast_density_ksi_per_mile']   = df_master['annual_forecast_ksi_crashes_2026']   / df_master['spatial_total_length_miles']

density_sorted = df_master.sort_values('forecast_density_total_per_mile', ascending=False).reset_index(drop=True)

print('── Top 10 High-Density Corridors (2026 Forecast Crashes per Mile) ──')
print(density_sorted[['corridor_id','corridor_name','spatial_total_length_miles','annual_forecast_total_crashes_2026','forecast_density_total_per_mile','forecast_density_ksi_per_mile']].head(10).to_string(index=False))


In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))

top15_density = density_sorted.head(15).iloc[::-1]
labels = [f"{r['corridor_name']} ({r['corridor_id']})" for _, r in top15_density.iterrows()]
values = top15_density['forecast_density_total_per_mile'].values

bars = ax.barh(labels, values, color='#e74c3c', alpha=0.85, edgecolor='white')
for bar, val in zip(bars, values):
    ax.text(bar.get_width() + 2, bar.get_y() + bar.get_height()/2,
            f'{val:.1f} / mi', va='center', fontsize=8)

ax.set_xlabel('2026 Forecast Annual Total Crashes per Mile')
ax.set_title('Top 15 Corridors by 2026 Forecast Crash Density (Crashes / Mile)', fontsize=11, fontweight='bold')
plt.tight_layout()
plt.show()


---
## Section 6 — Limitations

> **This notebook provides spatial GIS analytics for decision support.
> It does not constitute official City geometry approvals or engineering design.**

1. **100-foot buffer spatial assignment.** Crashes within 100 feet of a corridor centerline
   are assigned to the nearest candidate corridor. Boundary crashes near complex intersections
   may experience minor ambiguity (7.1% multi-candidate assignment rate).

2. **Corridor-level aggregate spatial resolution.** Crash density is calculated over
   the full corridor length. Sub-corridor segment bottlenecks or specific intersection
   hotspots are not isolated in this corridor-level panel.

3. **CDC/ATSDR SVI as project-defined planning proxy.** SVI scores measure census-tract
   socioeconomic vulnerability. This is an analyst-defined proxy for equity prioritization
   and does not represent the City of Chicago's official equity definition.

4. **Decision-support only.** Physical applicability (lane counts, median widths) remains
   UNKNOWN pending qualified engineering field survey.


In [ ]:
print('=' * 65)
print('SPATIAL ANALYSIS SUMMARY — Key Verified Findings')
print('=' * 65)
print(f'  Total corridors           : {len(df_master)}')
print(f'  Total network mileage     : {tot_miles:.2f} miles')
print(f'  Corridor length range     : {min_miles:.2f} to {max_miles:.2f} miles')
print(f'  High-SVI Corridors (A)    : {rule_a_count} / {len(df_master)} ({rule_a_count/len(df_master):.1%})')
print(f'  High-SVI Corridors (B)    : {rule_b_count} / {len(df_master)} ({rule_b_count/len(df_master):.1%})')
print()
print('  Top 3 Corridors by 2026 Forecast Crash Density:')
for idx, r in density_sorted.head(3).iterrows():
    print(f"    {idx+1}. {r['corridor_id']} {r['corridor_name']:<20} {r['forecast_density_total_per_mile']:>6.1f} crashes/mi  (Length: {r['spatial_total_length_miles']:.2f} mi)")
print('=' * 65)
